In [ ]:
import sys
sys.path.append("..")

import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from PIL import Image

import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

from model.upscaler import SuperResNet
from model.dataset import SuperResDataset
from model.lit_upscaler import LitSuperResNet
from model.upscaler_v0 import LitSuperResNetV0
from model.image_utils import ycbcr_tensor_to_pil

import matplotlib.pyplot as plt

from skimage.io import imread
import os

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
TEST_PATH = os.getenv("TEST_DATA_PATH")
_3RDPARTY_PATH = os.getenv("_3RDPARTY_PATH")

print(f"Test data path: {TEST_PATH}")

HIRES_PATCH_SIZE = 128

In [ ]:
upscaler_test = LitSuperResNet.load_from_checkpoint(
    '../model/checkpoints/v9_grad_lap_mae/upscaler-epoch=510.ckpt',
    ).to(DEVICE)
upscaler_test.eval()

upscaler_test_b = LitSuperResNetV0.load_from_checkpoint(
    '../model/checkpoints/v5_d3/0/upscaler-epoch=096.ckpt',
    ).to(DEVICE)
upscaler_test_b.eval()

In [ ]:
import torch.nn.functional as F

TEST_SEED = 42

test_dataset = SuperResDataset(
    TEST_PATH, 
    patch_size=HIRES_PATCH_SIZE,
    seed=TEST_SEED,
    downscale=None,
)
_3rdparty_dataset = SuperResDataset(
    _3RDPARTY_PATH, 
    patch_size=HIRES_PATCH_SIZE,
    seed=TEST_SEED,
    downscale=None,
)
test_loader = DataLoader(test_dataset,
                     batch_size=16, shuffle=False, num_workers=0)
_3rdparty_loader = DataLoader(_3rdparty_dataset,
                     batch_size=16, shuffle=False, num_workers=0)


_3rdparty_batch = next(iter(_3rdparty_loader))

batch = next(iter(test_loader))

with torch.no_grad():
    x, y_true = upscaler_test.batch_preprocess(batch)
    y_pred = upscaler_test(x.to(DEVICE)).clone()
    y_pred_b = upscaler_test_b(x.to(DEVICE)).clone() if upscaler_test_b is not None else None

for i in range(len(y_true)):

    lowres = ycbcr_tensor_to_pil(x[i].cpu())
    baseline = lowres.resize(
        (x.shape[-1] * 2, x.shape[-2] * 2),
        resample=Image.LANCZOS
    )
    
    panels = [
        (ycbcr_tensor_to_pil(y_pred[i]), "Pred"),
        (ycbcr_tensor_to_pil(y_true[i]), "GT"),
        (baseline, "LANCZOS"),
        (ycbcr_tensor_to_pil(_3rdparty_batch[i].cpu()), "3rd Party"),
    ]

    if y_pred_b is not None:
        panels.insert(1, (ycbcr_tensor_to_pil(y_pred_b[i]), "Pred B"))


    n = len(panels)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))

    if n == 1:
        axes = [axes]

    for ax, (img, title) in zip(axes, panels):
        ax.imshow(img)
        ax.set_title(title)
        ax.axis("off")

    plt.show()
